```text

Business Understanding
        ↓
Data Collection
        ↓
Dataset Understanding
        ↓
Data Cleaning (Remove missing values, duplicates, invalid text)
        ↓
Train / Validation / Test Split
        ↓
Text Preprocessing (If Required)
        ↓
Choose Pretrained Transformer Model
        ↓
Load Tokenizer
        ↓
Tokenization (Convert Text into Token IDs)
        ↓
Padding / Truncation / Attention Masks
        ↓
Create Dataset (PyTorch Dataset / Hugging Face Dataset)
        ↓
Create DataLoaders
        ↓
Visualize & Inspect Tokenized Samples
        ↓
Load Transformer Architecture
        ↓
Configure Task-Specific Output Head
        ↓
Choose Loss Function
        ↓
Choose Optimizer & Learning Rate Scheduler
        ↓
Training / Fine-Tuning Loop
        ↓
Validation
        ↓
Hyperparameter Tuning (Optional)
        ↓
Model Evaluation (Task-Specific Metrics)
        ↓
Error Analysis
        ↓
Save Trained Model + Tokenizer + Config
        ↓
Inference / Prediction
        ↓
Build Application (Gradio / Streamlit)
        ↓
Deployment (Hugging Face Spaces / Other Platform)
        ↓
GitHub + README

In [1]:

!pip install -q transformers torch sentencepiece


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch


In [3]:

%pip install -q "transformers<5"


In [4]:
from transformers import pipeline

summarizer = pipeline("summarization",model="sshleifer/distilbart-cnn-12-6")

print("Transformer model loaded successfully!")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Transformer model loaded successfully!


In [5]:

import transformers
import torch

print("Transformers version:", transformers.__version__)
print("Transformers location:", transformers.__file__)
print("PyTorch version:", torch.__version__)


Transformers version: 4.57.6
Transformers location: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py
PyTorch version: 2.11.0+cu130


In [6]:
print(type(summarizer.tokenizer))
print(type(summarizer.model))

<class 'transformers.models.bart.tokenization_bart_fast.BartTokenizerFast'>
<class 'transformers.models.bart.modeling_bart.BartForConditionalGeneration'>


In [7]:
#how to get all task identifiers

from transformers.pipelines import get_supported_tasks

tasks = get_supported_tasks()

for task in tasks:
    print(task)

print("Total tasks:", len(tasks))

audio-classification
automatic-speech-recognition
depth-estimation
document-question-answering
feature-extraction
fill-mask
image-classification
image-feature-extraction
image-segmentation
image-text-to-text
image-to-image
image-to-text
keypoint-matching
mask-generation
ner
object-detection
question-answering
sentiment-analysis
summarization
table-question-answering
text-classification
text-generation
text-to-audio
text-to-speech
text2text-generation
token-classification
translation
video-classification
visual-question-answering
vqa
zero-shot-audio-classification
zero-shot-classification
zero-shot-image-classification
zero-shot-object-detection
Total tasks: 34


In [8]:

print("Task:", summarizer.task)

print("Model architecture:", summarizer.model.config.model_type)

print("Tokenizer:", type(summarizer.tokenizer).__name__)

print("Model class:", type(summarizer.model).__name__)


Task: summarization
Model architecture: bart
Tokenizer: BartTokenizerFast
Model class: BartForConditionalGeneration


In [9]:
#providing it a article


article = """
Artificial Intelligence is transforming many industries,
including healthcare, education, transportation, and finance.
Machine learning enables computers to learn patterns from data,
while deep learning uses neural networks to solve complex tasks.

Transformer models have significantly improved natural language
processing. They use attention mechanisms to understand relationships
between tokens in a sequence. Models based on the Transformer
architecture can summarize documents, translate languages,
answer questions, and generate human-like text.
"""

print(article)



Artificial Intelligence is transforming many industries,
including healthcare, education, transportation, and finance.
Machine learning enables computers to learn patterns from data,
while deep learning uses neural networks to solve complex tasks.

Transformer models have significantly improved natural language
processing. They use attention mechanisms to understand relationships
between tokens in a sequence. Models based on the Transformer
architecture can summarize documents, translate languages,
answer questions, and generate human-like text.



In [12]:
#generate its summary


result = summarizer(article,max_length=60,min_length=20,do_sample=False)

print(result)



#article
#max_length = Sets the maximum generated sequence length, including special tokens
# min_length = Requests a minimum generated sequence length
#do sample =  Uses deterministic, non-sampling decoding, typically beam search for this model's default generation configuration

[{'summary_text': ' Machine learning enables computers to learn patterns from data, while deep learning uses neural networks to solve complex tasks . Models based on the Transformer architecture can summarize documents, translate languages, answer questions, and generate human-like text .'}]


### Common Hugging Face Pipeline & Generation Parameters

When calling a pipeline (like `summarizer`), you can customize the generation behavior using various parameters:

1. **Length Control**:
   * `max_length` (int): The maximum length of the sequence to be generated.
   * `min_length` (int): The minimum length of the sequence to be generated.
   * `max_new_tokens` (int): The maximum number of new tokens to generate, ignoring prompt tokens.
   * `min_new_tokens` (int): The minimum number of new tokens to generate.

2. **Generation Strategy**:
   * `do_sample` (bool): Whether or not to use sampling; use `False` for greedy decoding.
   * `num_beams` (int): Number of beams for beam search. `1` means no beam search.
   * `temperature` (float): Modulates next token probabilities (higher = more creative/random, lower = more deterministic).
   * `top_k` (int): Keep only the highest fold `k` filtering parameters.
   * `top_p` (float): Keep only the nucleus parameter (tokens summing up to cumulative probability `p`).

3. **Repetition Control**:
   * `repetition_penalty` (float): Penalty parameter for repeating tokens (1.0 means no penalty).
   * `no_repeat_ngram_size` (int): All ngrams of this size can only occur once.

In [14]:

summary = result[0]["summary_text"]

print("ORIGINAL ARTICLE:")
print(article)

print("\nGENERATED SUMMARY:")
print(summary)


ORIGINAL ARTICLE:

Artificial Intelligence is transforming many industries,
including healthcare, education, transportation, and finance.
Machine learning enables computers to learn patterns from data,
while deep learning uses neural networks to solve complex tasks.

Transformer models have significantly improved natural language
processing. They use attention mechanisms to understand relationships
between tokens in a sequence. Models based on the Transformer
architecture can summarize documents, translate languages,
answer questions, and generate human-like text.


GENERATED SUMMARY:
 Machine learning enables computers to learn patterns from data, while deep learning uses neural networks to solve complex tasks . Models based on the Transformer architecture can summarize documents, translate languages, answer questions, and generate human-like text .


In [15]:
#inspection

# Get the tokenizer attached to our pipeline
tokenizer = summarizer.tokenizer

# Convert the article into token IDs
tokens = tokenizer.tokenize(article)
token_ids = tokenizer.encode(article)

print("First 20 tokens:")
print(tokens[:20])

print("\nFirst 20 token IDs:")
print(token_ids[:20])

print("\nTotal token count:", len(token_ids))

First 20 tokens:
['Ċ', 'Art', 'ificial', 'ĠIntelligence', 'Ġis', 'Ġtransforming', 'Ġmany', 'Ġindustries', ',', 'Ċ', 'including', 'Ġhealthcare', ',', 'Ġeducation', ',', 'Ġtransportation', ',', 'Ġand', 'Ġfinance', '.']

First 20 token IDs:
[0, 50118, 23295, 37465, 6558, 16, 17828, 171, 4510, 6, 50118, 8529, 3717, 6, 1265, 6, 4264, 6, 8, 2879]

Total token count: 101


In [16]:

model = summarizer.model

print("Model type:", model.config.model_type)
print("Encoder layers:", model.config.encoder_layers)
print("Decoder layers:", model.config.decoder_layers)
print("Hidden size:", model.config.d_model)


Model type: bart
Encoder layers: 12
Decoder layers: 6
Hidden size: 1024


In [17]:

def summarize_text(text):
    result = summarizer(
        text,
        max_length=60,
        min_length=20,
        do_sample=False
    )

    return result[0]["summary_text"]


article = """
Artificial Intelligence is changing healthcare.
Machine learning helps computers learn from data.
Transformers are widely used for natural language processing.
"""

summary = summarize_text(article)

print("Summary:", summary)


Your max_length is set to 60, but your input_length is only 32. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)


Summary:  Transformers are widely used for natural language processing . Machine learning helps computers learn from data .


In [22]:
print(article)


Artificial Intelligence is changing healthcare.
Machine learning helps computers learn from data.
Transformers are widely used for natural language processing.



In [23]:
#final function


def summarize_text(text):
    input_tokens = len(tokenizer.encode(text, add_special_tokens=True))

    # Set limits based on the input length
    max_len = min(60, max(8, input_tokens - 2))
    min_len = min(10, max(1, max_len // 3))

    result = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )

    return result[0]["summary_text"]


summary = summarize_text(article)
print("Summary:", summary)


Summary:  Transformers are widely used for natural language processing . Machine learning helps computers learn from data .


In [24]:

long_article = """
Artificial Intelligence is transforming healthcare, education,
finance, and transportation. Machine learning allows computers
to identify patterns in large datasets. Deep learning uses
multi-layer neural networks to learn complex representations.

Transformer architectures have revolutionized natural language
processing through attention mechanisms. Models such as BERT,
GPT, and BART support tasks including text classification,
text generation, translation, and summarization.

Businesses increasingly use AI to automate repetitive tasks,
analyze information, and improve decision-making. However,
responsible AI development requires attention to privacy,
fairness, transparency, and reliability.
"""

print(summarize_text(long_article))


 Machine learning allows computers to identify patterns in large datasets . Deep learning uses multi-layer neural networks to learn complex representations . Businesses increasingly use AI to automate repetitive tasks and improve decision-making .
